# MacroAnchor G10 10Y Relative Value — V2.3 upload-first frozen implementation

This version fixes the execution workflow. **It does not auto-fetch FRED from the Colab runtime.**

## Before pressing Run all

Download these six fixed FRED bundles in your browser:

- [1 — USA + Germany](https://fred.stlouisfed.org/graph/fredgraph.csv?id=IRLTLT01USM156N%2CIR3TIB01USM156N%2CUSACPIALLMINMEI%2CCPIAUCSL%2CIRLTLT01DEM156N%2CIR3TIB01DEM156N%2CDEUCPIALLMINMEI&cosd=1960-01-01&coed=2026-05-31)
- [2 — UK + Japan](https://fred.stlouisfed.org/graph/fredgraph.csv?id=IRLTLT01GBM156N%2CIR3TIB01GBM156N%2CGBRCPIALLMINMEI%2CIRLTLT01JPM156N%2CIR3TIB01JPM156N%2CJPNCPIALLMINMEI&cosd=1960-01-01&coed=2026-05-31)
- [3 — Canada + Switzerland](https://fred.stlouisfed.org/graph/fredgraph.csv?id=IRLTLT01CAM156N%2CIR3TIB01CAM156N%2CCANCPIALLMINMEI%2CIRLTLT01CHM156N%2CIR3TIB01CHM156N%2CCHECPIALLMINMEI&cosd=1960-01-01&coed=2026-05-31)
- [4 — Sweden + Norway](https://fred.stlouisfed.org/graph/fredgraph.csv?id=IRLTLT01SEM156N%2CIR3TIB01SEM156N%2CSWECPIALLMINMEI%2CIRLTLT01NOM156N%2CIR3TIB01NOM156N%2CNORCPIALLMINMEI&cosd=1960-01-01&coed=2026-05-31)
- [5 — Australia](https://fred.stlouisfed.org/graph/fredgraph.csv?id=IRLTLT01AUM156N%2CIR3TIB01AUM156N%2CAUSCPIALLQINMEI&cosd=1960-01-01&coed=2026-05-31)
- [6 — New Zealand](https://fred.stlouisfed.org/graph/fredgraph.csv?id=IRLTLT01NZM156N%2CIR3TIB01NZM156N%2CNZLCPIALLQINMEI&cosd=1960-01-01&coed=2026-05-31)

FRED will usually call them `fredgraph.csv`, `fredgraph (1).csv`, etc. **Do not rename them; filenames do not matter.**

Then press **Run all**.

### What happens next

1. The **first operational cell immediately opens the Colab upload window**.
2. Select the six CSV files together.
3. After that, the notebook runs unattended. **There are no later upload prompts and no FRED network calls from Colab.**

You may alternatively upload a previously frozen `g10_panel_model_objects.csv` or `g10_panel_monthly_snapshot.csv` in that same first window.

## Frozen research choices

The parent signal remains unchanged: country-level MacroAnchor 10Y term-premium signal standardized using only past observations (`tp_z_past`).

- **Headline:** top-3 / bottom-3 countries.
- **Sensitivity only:** top-2 / bottom-2.
- **Second pre-specified implementation:** continuous cross-sectional score.
- Explicit net-DV01 neutrality.
- No new score family, threshold search, rebalance-frequency search, or ex-post country selection.

The strategy logic is unchanged from V2/V2.3. V2.3 changes only data ingestion and notebook UX.


In [ ]:
# ===== INPUT STEP — runs first and blocks immediately for upload =====
# Select ALL SIX fredgraph CSV files together.
# Alternative: upload g10_panel_model_objects.csv or g10_panel_monthly_snapshot.csv.
# After this cell completes, the notebook will not ask for any further input.

try:
    from google.colab import files
except ImportError as exc:
    raise RuntimeError("This notebook's upload-first workflow is intended for Google Colab.") from exc

print("UPLOAD NOW: select the six FRED fredgraph CSV files together.")
print("Alternative accepted inputs: g10_panel_model_objects.csv or g10_panel_monthly_snapshot.csv")
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No files uploaded. Upload the required input files before continuing.")

UPLOADED_NAMES = list(uploaded.keys())
print("\nUpload complete. From this point onward the notebook is unattended.")
print("Uploaded files:")
for name in UPLOADED_NAMES:
    print(" -", name)


In [ ]:
# ===== 0. Configuration =====
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import io, json, hashlib, warnings, math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.5f}")
plt.rcParams.update({"figure.dpi": 115, "font.size": 9, "axes.grid": True, "grid.alpha": 0.25})

START = pd.Timestamp("1960-01-01")
PANEL_END = pd.Timestamp("2026-05-31")
MINW = 120
H = 12
NW_LAG = 12
PI_HL = 36
RSTAR_HL = 120
ANCHOR_INITIALIZATION = "post_ewma_bfill"
QUARTERLY_CPI_RELEASE_LAG_MONTHS = 1

# Frozen strategy choices. Do not retune after seeing V2.3 results.
HEADLINE_K = 3
SENSITIVITY_K = 2
MIN_COUNTRIES = 6
COST_GRID_BP = [0.0, 1.0, 2.0, 5.0, 10.0, 25.0]
FX_HEDGE_ANNUAL_DRAG_BP = [0.0, 5.0, 10.0, 20.0]  # transparent stress only, not an estimate

MODEL_PANEL_FILENAME = "g10_panel_model_objects.csv"
MODEL_PANEL_EXPECTED_SHA = "1628c4eaff487f6d87cc93fe62c7af909850d9ceedf448d9d8321f8b8477bbf6"
SNAPSHOT_FILENAME = "g10_panel_monthly_snapshot.csv"
OUT_DIR = Path("macroanchor_g10_rv_v2_outputs")
OUT_DIR.mkdir(exist_ok=True)

# Data behavior: inputs come ONLY from the first upload window; no runtime FRED calls.

PANEL_SPEC = {
    "USA": {"y10": "IRLTLT01USM156N", "r3m": "IR3TIB01USM156N", "cpi": ["USACPIALLMINMEI", "CPIAUCSL"]},
    "DEU": {"y10": "IRLTLT01DEM156N", "r3m": "IR3TIB01DEM156N", "cpi": ["DEUCPIALLMINMEI"]},
    "GBR": {"y10": "IRLTLT01GBM156N", "r3m": "IR3TIB01GBM156N", "cpi": ["GBRCPIALLMINMEI"]},
    "JPN": {"y10": "IRLTLT01JPM156N", "r3m": "IR3TIB01JPM156N", "cpi": ["JPNCPIALLMINMEI"]},
    "CAN": {"y10": "IRLTLT01CAM156N", "r3m": "IR3TIB01CAM156N", "cpi": ["CANCPIALLMINMEI"]},
    "CHE": {"y10": "IRLTLT01CHM156N", "r3m": "IR3TIB01CHM156N", "cpi": ["CHECPIALLMINMEI"]},
    "SWE": {"y10": "IRLTLT01SEM156N", "r3m": "IR3TIB01SEM156N", "cpi": ["SWECPIALLMINMEI"]},
    "NOR": {"y10": "IRLTLT01NOM156N", "r3m": "IR3TIB01NOM156N", "cpi": ["NORCPIALLMINMEI"]},
    "AUS": {"y10": "IRLTLT01AUM156N", "r3m": "IR3TIB01AUM156N", "cpi": ["AUSCPIALLQINMEI"]},
    "NZL": {"y10": "IRLTLT01NZM156N", "r3m": "IR3TIB01NZM156N", "cpi": ["NZLCPIALLQINMEI"]},
}

print("Frozen V2.3 specification")
print("headline top/bottom:", HEADLINE_K, "| sensitivity:", SENSITIVITY_K)
print("panel cutoff:", PANEL_END.date(), "| forecast horizon:", H, "months")


In [ ]:
# ===== 1. Core model + inference helpers (same MacroAnchor panel logic) =====
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

def locate_file(name):
    for p in [Path(name), Path('/content')/name, Path('/mnt/data')/name]:
        if p.exists(): return p
    for root in [Path('/content'), Path.cwd(), Path('/mnt/data')]:
        if root.exists():
            hits = list(root.rglob(name))
            if hits: return hits[0]
    return None

def vas_B(tau, kappa):
    kappa = max(float(kappa), 1e-4)
    return (1.0 - np.exp(-kappa * tau)) / kappa

def vas_yield(tau, short_rate, kappa, theta, sigma):
    kappa = max(float(kappa), 1e-4)
    B = vas_B(tau, kappa)
    lnA = (theta - sigma*sigma/(2*kappa*kappa))*(B-tau) - sigma*sigma*B*B/(4*kappa)
    return (-lnA + B*short_rate) / tau

def build_anchor(df, initialization=ANCHOR_INITIALIZATION):
    idx = pd.DatetimeIndex(df.index)
    short_s = pd.Series(df['r3m'].to_numpy(float), index=idx)
    cpi_s = pd.Series(df['cpi'].to_numpy(float), index=idx)
    pi_raw = cpi_s.pct_change(12, fill_method=None).ewm(halflife=PI_HL).mean()
    if initialization == 'draft_pre_rstar_bfill':
        pi = pi_raw.bfill(); rstar = (short_s-pi).ewm(halflife=RSTAR_HL).mean().bfill()
    elif initialization in ('post_ewma_bfill','strict'):
        pi = pi_raw; rstar = (short_s-pi).ewm(halflife=RSTAR_HL).mean()
        if initialization == 'post_ewma_bfill':
            pi = pi.bfill(); rstar = rstar.bfill()
    else: raise ValueError(initialization)
    return rstar + pi, rstar, pi

def past_only_z(x, minobs=MINW):
    x = np.asarray(x, float); z = np.full(len(x), np.nan)
    for t in range(minobs, len(x)):
        hist = x[:t]; hist = hist[np.isfinite(hist)]
        if len(hist) < minobs or not np.isfinite(x[t]): continue
        sd = hist.std(ddof=1)
        if sd > 1e-12: z[t] = (x[t] - hist.mean()) / sd
    return z

def duration_proxy_10y(y):
    """Paper-consistent 10Y annuity/duration proxy used by the international target."""
    y = np.maximum(np.asarray(y, float), 1e-4)
    return (1.0 - (1.0 + y)**-10) / y

def run_country_objects(df):
    df = df.sort_index().copy(); n=len(df)
    r=df['r3m'].to_numpy(float); y10=df['y10'].to_numpy(float)
    anchor,rstar,pi=build_anchor(df); anc=anchor.to_numpy(float)
    kappa=np.full(n,np.nan); sigma=np.full(n,np.nan); tp=np.full(n,np.nan)
    for t in range(MINW,n):
        g=anc[:t]-r[:t]; dr=r[1:t+1]-r[:t]
        m=np.isfinite(g)&np.isfinite(dr)
        if m.sum()<MINW: continue
        X=np.column_stack([np.ones(m.sum()),g[m]])
        a0,slope=np.linalg.lstsq(X,dr[m],rcond=None)[0]
        kappa[t]=float(np.clip(slope*12.0,1e-4,20.0))
        resid=dr[m]-(a0+slope*g[m]); sigma[t]=float(np.std(resid,ddof=0)*np.sqrt(12.0))
        tp[t]=y10[t]-vas_yield(10.0,r[t],kappa[t],anc[t],sigma[t])
    D=duration_proxy_10y(y10)
    rx=np.full(n,np.nan)
    for t in range(n-H):
        rx[t]=y10[t]-D[t]*(y10[t+H]-y10[t])-r[t]
    return pd.DataFrame({
        'date':df.index,'country':df['country'].iloc[0],'y10':y10,'r3m':r,
        'slope':y10-r,'pi_trend':pi.to_numpy(float),'rstar':rstar.to_numpy(float),
        'anchor':anc,'kappa':kappa,'sigma':sigma,'tp':tp,'tp_z_past':past_only_z(tp),
        'duration10':D,'rx':rx
    })

def nw_fit(y, X=None, lag=NW_LAG):
    y=np.asarray(y,float)
    if X is None: X=np.empty((len(y),0))
    X=np.asarray(X,float)
    if X.ndim==1: X=X[:,None]
    m=np.isfinite(y)&np.isfinite(X).all(axis=1)
    y=y[m]; X=X[m]
    X=np.column_stack([np.ones(len(y)),X])
    b=np.linalg.lstsq(X,y,rcond=None)[0]; e=y-X@b
    bread=np.linalg.pinv(X.T@X); score=X*e[:,None]; S=score.T@score
    for ell in range(1,min(lag,len(y)-1)+1):
        w=1-ell/(lag+1); G=score[ell:].T@score[:-ell]; S+=w*(G+G.T)
    V=bread@S@bread; se=np.sqrt(np.maximum(np.diag(V),0))
    den=np.sum((y-y.mean())**2); r2=1-np.sum(e*e)/den if den>0 else np.nan
    return b,se,V,r2,int(len(y))

In [ ]:
# ===== 2. Upload-only data layer: model panel -> snapshot -> six browser-downloaded FRED bundles =====
# IMPORTANT: no filesystem search and no web request occurs here.
# Only files explicitly selected in the first upload window are eligible inputs.

def monthly_cpi_from_source(cpi, series_id):
    cpi = cpi.dropna().sort_index().astype(float)
    if 'QINMEI' in str(series_id).upper():
        qend = cpi.index.to_period('Q').to_timestamp(how='end').normalize()
        release = qend + pd.offsets.MonthEnd(QUARTERLY_CPI_RELEASE_LAG_MONTHS)
        s = pd.Series(cpi.to_numpy(), index=release).groupby(level=0).last().sort_index()
        return s.resample('ME').last().ffill()
    return cpi.resample('ME').last()

def required_fred_series():
    ids = []
    for spec in PANEL_SPEC.values():
        ids.extend([spec['y10'], spec['r3m']])
        ids.extend(spec['cpi'])
    return list(dict.fromkeys(ids))

def _uploaded_path(name):
    p = Path(name)
    if not p.exists():
        raise FileNotFoundError(f"Uploaded file is not present in the Colab working directory: {name}")
    return p

def _read_candidate_fred_csv(name):
    """Read one uploaded CSV; return only recognized canonical FRED columns."""
    p = _uploaded_path(name)
    try:
        raw = pd.read_csv(p)
    except Exception:
        return None

    if raw.empty or raw.shape[1] < 2:
        return None

    required = set(required_fred_series())
    matching = [c for c in raw.columns if c in required]
    if not matching:
        return None

    date_col = next(
        (c for c in raw.columns if str(c).strip().lower() in {'observation_date','date'}),
        raw.columns[0],
    )
    dates = pd.to_datetime(raw[date_col], errors='coerce')
    keep = dates.notna()

    out = raw.loc[keep, matching].copy()
    out.index = pd.DatetimeIndex(dates.loc[keep])
    out.index.name = 'date'
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors='coerce')
    return out.sort_index()

def merge_uploaded_fred_bundles():
    candidates = []
    for name in UPLOADED_NAMES:
        if not str(name).lower().endswith('.csv'):
            continue
        if Path(name).name in {MODEL_PANEL_FILENAME, SNAPSHOT_FILENAME}:
            continue
        frame = _read_candidate_fred_csv(name)
        if frame is not None:
            candidates.append((name, frame))

    if not candidates:
        raise RuntimeError(
            "None of the uploaded files contains the canonical FRED series. "
            "Upload the six fredgraph CSV files linked at the top of the notebook."
        )

    print("Recognized uploaded FRED bundles:")
    for name, frame in candidates:
        print(f"  {name}: {len(frame.columns)} canonical series")

    merged = pd.DataFrame()
    for name, frame in candidates:
        if merged.empty:
            merged = frame.copy()
            continue

        idx = merged.index.union(frame.index)
        merged = merged.reindex(idx)
        frame = frame.reindex(idx)

        for col in frame.columns:
            if col not in merged.columns:
                merged[col] = frame[col]
            else:
                both = merged[col].notna() & frame[col].notna()
                if both.any():
                    diff = (merged.loc[both, col] - frame.loc[both, col]).abs()
                    if (diff > 1e-12).any():
                        bad = diff[diff > 1e-12].index[0]
                        raise ValueError(f"Conflicting uploaded values for {col} at {bad.date()}.")
                merged[col] = merged[col].combine_first(frame[col])

    # Validate country-by-country rather than demanding both alternative US CPI series.
    missing = {}
    for country, spec in PANEL_SPEC.items():
        miss = []
        for sid in [spec['y10'], spec['r3m']]:
            if sid not in merged.columns or merged[sid].notna().sum() < 24:
                miss.append(sid)
        if not any(sid in merged.columns and merged[sid].notna().sum() >= 24 for sid in spec['cpi']):
            miss.append("one_of:" + "|".join(spec['cpi']))
        if miss:
            missing[country] = miss

    if missing:
        raise RuntimeError(
            "Uploaded FRED bundles are incomplete. Missing canonical inputs:\n"
            + json.dumps(missing, indent=2)
        )

    print("Uploaded FRED coverage validation: PASS")
    return merged.sort_index()

def build_raw_snapshot_from_uploaded_bundles():
    wide = merge_uploaded_fred_bundles()
    rows = []
    failures = {}

    for country, spec in PANEL_SPEC.items():
        try:
            cpi_id = next(
                sid for sid in spec['cpi']
                if sid in wide.columns and wide[sid].notna().sum() >= 24
            )

            y = wide[spec['y10']].dropna().resample('ME').last() / 100.0
            r = wide[spec['r3m']].dropna().resample('ME').last() / 100.0
            cp = monthly_cpi_from_source(wide[cpi_id], cpi_id)

            d = pd.concat(
                [y.rename('y10'), r.rename('r3m'), cp.rename('cpi')],
                axis=1
            )
            d = d.loc[(d.index >= START) & (d.index <= PANEL_END)].dropna()

            if len(d) < MINW + H + 24:
                raise RuntimeError(f"only {len(d)} complete months")

            z = d.reset_index()
            z = z.rename(columns={z.columns[0]: 'date'})
            z.insert(1, 'country', country)
            rows.append(z)
            print(country, "OK", len(d), d.index.min().date(), "->", d.index.max().date())

        except Exception as exc:
            failures[country] = repr(exc)
            print(country, "FAILED", exc)

    if len(rows) < MIN_COUNTRIES:
        raise RuntimeError(
            "Uploaded data did not assemble enough countries.\n"
            + json.dumps(failures, indent=2)
        )

    raw = pd.concat(rows, ignore_index=True).sort_values(['country','date'])
    raw.to_csv(SNAPSHOT_FILENAME, index=False)
    print(f"Frozen raw snapshot written to {SNAPSHOT_FILENAME}")
    return raw

def load_model_panel():
    uploaded_basenames = {Path(x).name: x for x in UPLOADED_NAMES}

    # 1) Exact frozen model panel uploaded in the initial window.
    if MODEL_PANEL_FILENAME in uploaded_basenames:
        path = _uploaded_path(uploaded_basenames[MODEL_PANEL_FILENAME])
        got = sha256_file(path)
        if got == MODEL_PANEL_EXPECTED_SHA:
            print("Using uploaded frozen V7 model panel; SHA verified.")
        else:
            warnings.warn(
                "Uploaded model-panel SHA differs from archived V7 hash; "
                "continuing but recording provenance."
            )
        p = pd.read_csv(path, parse_dates=['date']).sort_values(['country','date']).reset_index(drop=True)
        if 'duration10' not in p:
            p['duration10'] = duration_proxy_10y(p['y10'])
        return p, {'source':'uploaded_model_panel','path':str(path),'sha256':got}

    # 2) Exact frozen raw snapshot uploaded in the initial window.
    if SNAPSHOT_FILENAME in uploaded_basenames:
        path = _uploaded_path(uploaded_basenames[SNAPSHOT_FILENAME])
        print("Using uploaded frozen raw snapshot:", path)
        raw = pd.read_csv(path, parse_dates=['date']).sort_values(['country','date'])
        source = {'source':'uploaded_raw_snapshot','path':str(path),'sha256':sha256_file(path)}

    # 3) Otherwise the six browser-downloaded FRED bundles from the same upload window.
    else:
        print("No frozen panel/snapshot uploaded; building from the uploaded six FRED bundles.")
        raw = build_raw_snapshot_from_uploaded_bundles()
        source = {
            'source':'uploaded_fred_six_bundle',
            'path':SNAPSHOT_FILENAME,
            'sha256':sha256_file(SNAPSHOT_FILENAME),
        }

    objs = []
    for country, g in raw.groupby('country'):
        d = g.sort_values('date').set_index('date').copy()
        d['country'] = country
        objs.append(run_country_objects(d))

    p = pd.concat(objs, ignore_index=True).sort_values(['country','date']).reset_index(drop=True)
    p.to_csv(MODEL_PANEL_FILENAME, index=False)
    source['generated_model_panel_sha256'] = sha256_file(MODEL_PANEL_FILENAME)
    return p, source

model_panel, DATA_PROVENANCE = load_model_panel()
print('rows:', len(model_panel), '| raw countries:', model_panel.country.nunique())
analysis = model_panel.dropna(subset=['rx','tp_z_past','y10']).copy()
print('usable:', len(analysis), 'country-months |', analysis.country.nunique(), 'countries |', analysis.date.nunique(), 'months')
print('provenance:', DATA_PROVENANCE)


In [ ]:
# ===== 3. Pure strategy constructors: explicit DV01 neutrality =====
def _normalize_notional_from_dv01_targets(g, q):
    """q = desired signed DV01 units. Converts to notionals and rescales gross notional to 1."""
    D=g['duration10'].to_numpy(float); q=np.asarray(q,float)
    w=q/D
    gross=np.sum(np.abs(w))
    if gross<=1e-12: return np.full(len(g),np.nan)
    w=w/gross
    # Hard identity: net DV01 must be zero up to numerical precision.
    assert abs(np.sum(w*D)) < 1e-10
    return w

def construct_rank_weights(g,k):
    g=g.dropna(subset=['tp_z_past','duration10']).sort_values('tp_z_past').copy()
    if len(g)<max(MIN_COUNTRIES,2*k): return None
    q=np.zeros(len(g)); q[:k]=-1.0/k; q[-k:]=1.0/k
    w=_normalize_notional_from_dv01_targets(g,q)
    g['weight']=w; g['dv01_weight']=w*g['duration10']
    return g

def construct_continuous_weights(g):
    g=g.dropna(subset=['tp_z_past','duration10']).copy()
    if len(g)<MIN_COUNTRIES: return None
    s=g['tp_z_past'].to_numpy(float); q=s-np.mean(s)
    if np.sum(np.abs(q))<1e-12: return None
    q=q/np.sum(np.abs(q))
    w=_normalize_notional_from_dv01_targets(g,q)
    g['weight']=w; g['dv01_weight']=w*g['duration10']
    return g

def form_portfolio_panel(panel, mode='rank', k=HEADLINE_K, signal='tp_z_past', target='rx'):
    rows=[]; weight_rows=[]
    for date,g0 in panel.groupby('date'):
        cols=['date','country','y10','r3m','slope','duration10',signal,target]
        g=g0[[c for c in cols if c in g0.columns]].dropna(subset=[signal,target,'duration10']).copy()
        # Map any requested signal into the constructor's canonical score column exactly once.
        # Do NOT rename the original signal afterward: that would create duplicate 'tp_z_past' columns.
        if signal!='tp_z_past':
            g['tp_z_past']=g[signal].astype(float)
        if mode=='rank':
            z=construct_rank_weights(g,k)
        elif mode=='continuous':
            z=construct_continuous_weights(g)
        else: raise ValueError(mode)
        if z is None: continue
        # recover the requested signal name if constructor renamed it
        ret=float(np.sum(z['weight']*z[target]))
        sig=float(np.sum(z['weight']*z['tp_z_past']))
        rows.append({'date':date,'ret12':ret,'signal_spread':sig,'n_countries':len(z),
                     'gross_notional':float(z['weight'].abs().sum()),
                     'net_notional':float(z['weight'].sum()),
                     'net_dv01':float(z['dv01_weight'].sum())})
        for _,r in z.iterrows():
            weight_rows.append({'date':date,'country':r['country'],'weight':r['weight'],
                                'dv01_weight':r['dv01_weight'],'duration10':r['duration10'],
                                'score':r['tp_z_past']})
    return pd.DataFrame(rows).set_index('date').sort_index(), pd.DataFrame(weight_rows)

def weight_turnover(weight_rows):
    W=weight_rows.pivot(index='date',columns='country',values='weight').fillna(0).sort_index()
    # one-way turnover: half of gross target-weight change
    to=0.5*W.diff().abs().sum(axis=1)
    if len(to): to.iloc[0]=0.5*W.iloc[0].abs().sum()
    return W,to

def summary_12m(port, weight_rows, cost_bp=0.0, fx_drag_bp=0.0):
    x=port['ret12'].dropna(); b,se,V,r2,n=nw_fit(x.to_numpy(),None,NW_LAG)
    W,to=weight_turnover(weight_rows)
    ann_turnover=float(12*to.reindex(x.index).mean())
    gross=float(x.mean())                     # each observation is already a 12m return
    net=gross-ann_turnover*(cost_bp/1e4)-fx_drag_bp/1e4
    tracks=[]
    for m in range(1,13):
        a=x[x.index.month==m]
        if len(a)>=8 and a.std(ddof=1)>1e-12: tracks.append(float(a.mean()/a.std(ddof=1)))
    return {'months':n,'gross_ann_mean':gross,'NW_t':float(b[0]/se[0]),
            'median_nonoverlap_SR':float(np.median(tracks)) if tracks else np.nan,
            'positive_tracks':f"{sum(t>0 for t in tracks)}/{len(tracks)}",
            'ann_turnover':ann_turnover,'net_ann_mean':net,
            'breakeven_bp':gross*1e4/ann_turnover if ann_turnover>0 else np.nan,
            'avg_abs_net_dv01':float(port['net_dv01'].abs().mean()),
            'avg_abs_net_notional':float(port['net_notional'].abs().mean())}


In [ ]:
# ===== 4. Build the two frozen V2 portfolios =====
headline, W_head = form_portfolio_panel(analysis,mode='rank',k=HEADLINE_K)
sens2, W_sens2 = form_portfolio_panel(analysis,mode='rank',k=SENSITIVITY_K)
continuous, W_cont = form_portfolio_panel(analysis,mode='continuous')

rows=[]
for name,p,w in [('rank_top3_bottom3_DV01',headline,W_head),('rank_top2_bottom2_DV01_sensitivity',sens2,W_sens2),('continuous_score_DV01',continuous,W_cont)]:
    s=summary_12m(p,w,0.0,0.0); rows.append({'strategy':name,**s})
perf=pd.DataFrame(rows).set_index('strategy')
print('Gross V2.3 results (12m overlapping formation returns; NW(12) inference)')
display(perf.round(5))

# Hard neutrality diagnostics
assert perf['avg_abs_net_dv01'].max() < 1e-10
print('DV01 neutrality identity: PASS')

In [ ]:
# ===== 5. Cost and FX-hedge stress grid =====
cost_rows=[]
for name,p,w in [('rank_top3_bottom3_DV01',headline,W_head),('continuous_score_DV01',continuous,W_cont)]:
    for c in COST_GRID_BP:
        for fx in FX_HEDGE_ANNUAL_DRAG_BP:
            s=summary_12m(p,w,c,fx)
            cost_rows.append({'strategy':name,'oneway_cost_bp':c,'fx_extra_annual_drag_bp':fx,
                              'gross_ann_mean':s['gross_ann_mean'],'net_ann_mean':s['net_ann_mean'],
                              'ann_turnover':s['ann_turnover'],'breakeven_bp':s['breakeven_bp']})
cost_table=pd.DataFrame(cost_rows)
display(cost_table.query("fx_extra_annual_drag_bp==0").round(5))

In [ ]:
# ===== 6. Pre-specified simple benchmarks: global duration, yield-level RV, slope/carry RV =====
# Past-only controls are the same objects used in notebook 09b.
for col in ['y10','slope']:
    if col+'_z_past' not in analysis:
        analysis[col+'_z_past']=analysis.groupby('country',group_keys=False)[col].transform(
            lambda s: pd.Series(past_only_z(s.to_numpy()),index=s.index))

# Global 10Y factor = equal-notional average of local-currency 10Y excess returns.
global10=analysis.groupby('date')['rx'].mean().rename('global10')

# Naive cross-country carry/value benchmarks use the exact same DV01-neutral rank constructor,
# changing only the signal. No benchmark tuning.
yield_ls, W_yield = form_portfolio_panel(analysis,mode='rank',k=HEADLINE_K,signal='y10_z_past')
slope_ls, W_slope = form_portfolio_panel(analysis,mode='rank',k=HEADLINE_K,signal='slope_z_past')

bench=pd.concat([global10,yield_ls['ret12'].rename('yield_level_LS'),slope_ls['ret12'].rename('slope_carry_LS')],axis=1)
display(bench.describe().T[['mean','std','count']])


In [ ]:
# ===== 7. Spanning / attribution regressions =====
def spanning(y, Xdf, label):
    d=pd.concat([y.rename('y'),Xdf],axis=1).dropna()
    b,se,V,r2,n=nw_fit(d['y'].to_numpy(),d[Xdf.columns].to_numpy(),NW_LAG)
    out={'model':label,'alpha_ann':float(b[0]),'alpha_t_NW':float(b[0]/se[0]),'R2':r2,'n':n}
    for j,c in enumerate(Xdf.columns,1): out['beta_'+c]=float(b[j])
    return out

span_rows=[]
for nm,p in [('rank_top3_bottom3_DV01',headline),('continuous_score_DV01',continuous)]:
    y=p['ret12']
    span_rows.append(spanning(y,bench[['global10']],nm+' ~ global10'))
    span_rows.append(spanning(y,bench[['global10','yield_level_LS','slope_carry_LS']],nm+' ~ global10 + yield/slope RV'))
spanning_table=pd.DataFrame(span_rows).set_index('model')
display(spanning_table.round(5))

print('Reading rule: positive alpha after global-duration and naive carry/value spanning is incremental MacroAnchor evidence;')
print('a null alpha means the tradable spread is explained by familiar sovereign-rate exposures even if the predictor slope remains significant.')

In [ ]:
# ===== 8. Stability by pre-specified regimes =====
def regime(d):
    d=pd.Timestamp(d)
    if d<=pd.Timestamp('2007-12-31'): return 'pre_2008'
    if d<=pd.Timestamp('2015-12-31'): return '2008_2015'
    return '2016_plus'

sub_rows=[]
for nm,p,w in [('rank_top3_bottom3_DV01',headline,W_head),('continuous_score_DV01',continuous,W_cont)]:
    for reg,g in p.assign(regime=[regime(x) for x in p.index]).groupby('regime'):
        ww=w[w['date'].isin(g.index)]
        s=summary_12m(g.drop(columns='regime'),ww,0.0,0.0)
        sub_rows.append({'strategy':nm,'regime':reg,**s})
subperiods=pd.DataFrame(sub_rows).set_index(['strategy','regime'])
display(subperiods[['months','gross_ann_mean','NW_t','median_nonoverlap_SR','ann_turnover']].round(5))

In [ ]:
# ===== 9. Diagnostics: country concentration, score dispersion, neutrality =====
def concentration(weight_rows):
    W=weight_rows.pivot(index='date',columns='country',values='weight').fillna(0).sort_index()
    hhi=(W.abs().div(W.abs().sum(axis=1),axis=0).pow(2).sum(axis=1))
    avg_abs=W.abs().mean().sort_values(ascending=False)
    return W,hhi,avg_abs

Wmat,hhi,country_exposure=concentration(W_head)
print('Headline median HHI:',float(hhi.median()),'| max:',float(hhi.max()))
display(country_exposure.rename('avg_abs_notional_weight').to_frame())

fig,ax=plt.subplots(1,2,figsize=(12,4))
headline['ret12'].dropna().cumsum().plot(ax=ax[0],title='Headline DV01-neutral cross-country spread: cumulative 12m-formation mean path')
ax[0].set_ylabel('cumulative spread (not a live wealth index)')
country_exposure.sort_values().plot.barh(ax=ax[1],title='Average absolute country notional weight')
plt.tight_layout(); plt.savefig(OUT_DIR/'figure_headline_diagnostics.png',bbox_inches='tight'); plt.show()

In [ ]:
# ===== 10. Export deterministic research bundle =====
perf.to_csv(OUT_DIR/'v2_performance.csv')
cost_table.to_csv(OUT_DIR/'v2_cost_fx_stress.csv',index=False)
spanning_table.to_csv(OUT_DIR/'v2_spanning.csv')
subperiods.to_csv(OUT_DIR/'v2_subperiods.csv')
W_head.to_csv(OUT_DIR/'v2_weights_headline.csv',index=False)
W_cont.to_csv(OUT_DIR/'v2_weights_continuous.csv',index=False)
country_exposure.to_csv(OUT_DIR/'v2_country_exposure.csv',header=['avg_abs_notional_weight'])

manifest={
    'label':'macroanchor_g10_10y_rv_v2',
    'panel_cutoff':str(PANEL_END.date()),
    'parent_signal':'MacroAnchor 10Y tp_z_past; unchanged',
    'headline':'top-3/bottom-3 explicit DV01-neutral',
    'sensitivity':'top-2/bottom-2 only; inherited from notebook 10',
    'second_spec':'continuous demeaned-score explicit DV01-neutral',
    'inference':'Newey-West lag 12 on overlapping 12m formation returns; 12 non-overlapping calendar-track Sharpe diagnostic',
    'fx_interpretation':'local excess return equals FX-hedged excess over base cash under CIP to first order; basis/forward execution not observed',
    'data_provenance':DATA_PROVENANCE,
    'no_search_statement':'No new predictor family, k-grid, threshold grid, or rebalance-frequency search introduced in V2.'
}
(OUT_DIR/'v2_manifest.json').write_text(json.dumps(manifest,indent=2,default=str))
print('Exported to',OUT_DIR.resolve())

## Decision hierarchy

Do **not** select among V2 strategies by whichever Sharpe is largest.

1. `rank_top3_bottom3_DV01` is the headline because top-3/bottom-3 was fixed before V2.
2. `rank_top2_bottom2_DV01_sensitivity` is reported only as inherited sensitivity.
3. `continuous_score_DV01` is a second pre-specified implementation, not a challenger chosen after results.
4. The claim is strengthened only if the headline remains positive after explicit DV01 balancing, cost stress, subperiod decomposition, and spanning against global duration plus naive yield/slope RV benchmarks.
5. If the headline fails, report the failure. Do not reopen k, thresholds, score transforms, or country subsets.